# Part 14: Pairwise Bootstrap Inference

This notebook runs the reproducible Part 14 Python runner in Colab. Part 14 reads Part 10 outputs and applies circular moving-block bootstrap inference to core target-weight pairwise comparisons. It does not re-estimate PCA/HMM models and does not re-select allocation rules.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path
import os

PROJECT_DIR = Path('/content/drive/MyDrive/project_edhec_paper')
os.chdir(PROJECT_DIR)
print('Working directory:', Path.cwd())

assert (PROJECT_DIR / 'data_2026' / 'cleaned').exists(), 'Cleaned input folder not found.'
assert (PROJECT_DIR / 'experiments' / 'part14_pairwise_inference' / 'run_part14_pairwise_inference.py').exists(), 'Part 14 runner not found.'

In [ ]:
!python -m pip install -q -r experiments/part14_pairwise_inference/requirements-part14.txt

## Path configuration

Part 14 requires a completed Part 10 run. Run Part 10 first if `PART10_RUN_DIR` does not exist.

In [ ]:
from pathlib import Path

def choose_existing(*candidates):
    for candidate in candidates:
        path = Path(candidate)
        if path.exists():
            return path
    return Path(candidates[0])

INPUT_DIR = Path('data_2026/cleaned')
PART10_RUN_DIR = choose_existing(
    'outputs/part10_benchmark_cap_sensitivity/colab_part10_seed42',
    'outputs/part10_benchmark_cap_sensitivity_outputs/part10_benchmark_cap_sensitivity/colab_part10_seed42',
)
OUTPUT_DIR = Path('outputs/part14_pairwise_inference_outputs/part14_pairwise_inference')
RUN_ID = 'colab_part14_seed42'

print('INPUT_DIR:', INPUT_DIR)
print('PART10_RUN_DIR:', PART10_RUN_DIR)
print('OUTPUT_DIR:', OUTPUT_DIR)
assert PART10_RUN_DIR.exists(), 'Part 10 run directory not found. Run Part 10 first.'

In [ ]:
!python experiments/part14_pairwise_inference/run_part14_pairwise_inference.py \
  --input-dir "{INPUT_DIR}" \
  --part10-run-dir "{PART10_RUN_DIR}" \
  --output-dir "{OUTPUT_DIR}" \
  --run-id "{RUN_ID}" \
  --seed 42 \
  --bootstrap-reps 2000 \
  --block-length 13

In [ ]:
import json
import pandas as pd

RUN_DIR = OUTPUT_DIR / RUN_ID
RESULTS = RUN_DIR / 'results'
FIGURES = RUN_DIR / 'figures'

validation = json.loads((RESULTS / 'output_validation_summary.json').read_text())
print(json.dumps(validation, indent=2))
assert validation['status'] == 'passed'

In [ ]:
ci = pd.read_csv(RESULTS / 'part14_pairwise_bootstrap_ci.csv')
core = ci[
    ci['comparison_id'].str.contains('conditional_cap_10pct_vs_(?:matched_fixed_btc|cap_only_10pct)', regex=True)
    & ci['metric'].isin(['annualized_mean_arithmetic', 'annualized_volatility', 'btc_share_vol', 'btc_share_cvar'])
]
display(core[['comparison_id', 'metric', 'original_difference_left_minus_right', 'ci95_lower', 'ci95_upper', 'interpretation']])
display(pd.read_csv(RESULTS / 'part14_cap_exceedance_under_pairwise_bootstrap.csv'))
display(pd.read_csv(RESULTS / 'part14_inference_decision_matrix.csv').head(20))

In [ ]:
from IPython.display import Image, display

for name in [
    'part14_pairwise_ci_core_metrics.png',
    'part14_pairwise_ci_risk_contribution.png',
]:
    print(name)
    display(Image(filename=str(FIGURES / name)))

In [ ]:
import shutil

zip_path = shutil.make_archive(str(RUN_DIR), 'zip', root_dir=RUN_DIR)
print('Created zip:', zip_path)